# Step 34 — RavKav 2025: boardings by stop and TAZ, bus and Metronit OD with the OnBoard pattern, rail OD from entry and exit taps

Three RavKav (smart-card) extracts for 2025 were added on 23 September 2026 under
`Input/BusRavKav/2025/` — `Metronit_RavKav_Data.csv` (the Haifa BRT), `Rail_RavKav_Data.csv`
(every Israel Railways station) and `Buses_RavKav.csv` (a national all-modes extract, "mostly
buses") — together with `Input/BusRavKav/Stops_In_North/stops_in_taz_north.csv`, the stop codes
that lie inside the `TAZ_North` polygons. Every row is **one boarding tap**: card, operator
cluster, stop, date, time, a **transfer tag** (`מעבר` = a transfer boarding within a journey,
`לא מעבר` = a journey's first boarding), a passenger count and, on some rows, coordinates.
Weekday 3 (Tuesday), taps between 06:00 and 08:59, 51–52 Tuesdays across 2025.

The extracts hold **boardings only** (no alightings, no linked journeys — unlike the May 2022
files of step 8). This step therefore:

1. tags every stop with its TAZ (the north stops file, joined to the `TAZ_North` polygons);
2. **averages the many dates to one representative Tuesday**, after dropping the dates whose
   volume shows a holiday, the June war or a data gap;
3. keeps the **transfer and non-transfer boardings apart**: non-transfer boardings are
   journey origins (the unit of the OD), transfer boardings are the further legs of those
   journeys (the unit of a link load or a hub count);
4. builds the **bus + Metronit OD** by distributing each TAZ's journey origins over the
   OnBoard survey's P(alight | board) pattern — the construction of step 9, on 2025 volumes;
5. builds the **rail OD directly**: the rail rows with `PassengersNumber = 0` are the exit-gate
   taps, and pairing each card's entry with its next exit gives a station-to-station matrix
   (tested in §5 below), expanded to the entries whose exit falls after 09:00; the stations are
   located from the GTFS stops table, because the extract's own station coordinates are wrong
   for several stations;
6. compares the 2025 layer with the May 2022 RavKav products and the survey-based 2022 layers,
   and derives the **boarding-hour peak factors** that task B1e asked for.

Outputs go to `Output/ravkav_2025/`. Nothing downstream is re-pointed at them here; §7 says
what re-anchoring the chain on this layer would involve.

In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

In [2]:
import warnings, time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9'
OUT = 'Output/ravkav_2025'; os.makedirs(OUT, exist_ok=True); os.makedirs('Output/figures', exist_ok=True)
RK = 'Input/BusRavKav/2025'
STOPS_NORTH = 'Input/BusRavKav/Stops_In_North/stops_in_taz_north.csv'
USECOLS = ['CardIDbi', 'ClusterName', 'StopCode', 'StopName', 'TransactionDate', 'TransactionTime', 'JourneyTransfer', 'PassengersNumber', 'Lat', 'Long', 'Weekday']
TRANSFER, NONTRANSFER = 'מעבר', 'לא מעבר'
RAIL_CLUSTER, METRONIT_CLUSTER = 'רכבת ישראל', 'מטרונית חיפה'
LRT_CLUSTERS = {'קו אדום גוש דן', 'רכבת קלה ירושלים'}            # Tel Aviv Red Line, Jerusalem LRT — in the national file, not in the north
REP_DAY_FLOOR = 0.70                                              # a date is representative if its net boardings reach 70 % of the dataset's median date
HOURS = ['06', '07', '08']
def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
for f in [f'{RK}/Metronit_RavKav_Data.csv', f'{RK}/Rail_RavKav_Data.csv', f'{RK}/Buses_RavKav.csv', STOPS_NORTH]:
    assert not is_pointer(f), f'{f} is a Git LFS pointer — git lfs pull --include="Input/BusRavKav/2025/*,Input/BusRavKav/Stops_In_North/*" first'

## 1. Stops → TAZ, keyed by operator cluster and stop code

`stops_in_taz_north.csv` is a 25-million-row dump of `StopCode, Lat, Long` (one row per tap,
apparently) whose unique codes all fall inside the `TAZ_North` polygons. But **`StopCode` is not
unique across operators in these extracts**: on the rows that carry coordinates, Tel Aviv and
Jerusalem clusters tap "northern" codes at places far outside the study area (99.99 % of the
Tel Aviv cluster's coordinate rows with a code from the north file lie outside the north), and
the north file inherited those collisions. A stop is therefore keyed by **(cluster, code)**:

- its location is the coordinates the extracts themselves carry for that (cluster, code) — a
  quarter of the Metronit rows, a third of the bus rows and, per station, the rail rows have
  them — joined to the `TAZ_North` polygons exactly as step 8 tagged the 2022 stops;
- a (cluster, code) that never carries coordinates takes the north file's location **only if
  the cluster is a northern operator** (most of its located taps inside the polygons); otherwise
  it is dropped and counted.

In [3]:
t0 = time.time(); u = {}
for ch in pd.read_csv(STOPS_NORTH, chunksize=3_000_000):
    g = ch.dropna().drop_duplicates('StopCode')
    for code, la, lo in g[['StopCode', 'Lat', 'Long']].itertuples(index=False):
        if code not in u: u[code] = (la, lo)
north_file = pd.DataFrame([(k, v[0], v[1]) for k, v in u.items()], columns=['StopCode', 'Lat', 'Long']); north_file = north_file[north_file['StopCode'] > 0].set_index('StopCode')
taz = gpd.read_file('Input/TAZ_North/TAZ_North.shp')[['TAZ_NUMBER', 'geometry']]
def tag_points(df):
    pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['Long'], df['Lat']), crs='EPSG:4326').to_crs(taz.crs)
    return gpd.sjoin(pts, taz, how='left', predicate='within')['TAZ_NUMBER']
north_file['TAZ'] = tag_points(north_file.reset_index()).values
print(f'north stops file: {len(north_file):,} unique codes, {north_file["TAZ"].notna().sum():,} inside a TAZ_North polygon  [{time.time() - t0:.0f} s]')

# coordinates per (cluster, code) from the extracts themselves, and each cluster's share of located taps inside the polygons
t0 = time.time(); loc = {}; cl_rows = {}
for path in [f'{RK}/Buses_RavKav.csv', f'{RK}/Metronit_RavKav_Data.csv', f'{RK}/Rail_RavKav_Data.csv']:
    for ch in pd.read_csv(path, usecols=['ClusterName', 'StopCode', 'StopName', 'Lat', 'Long'], chunksize=2_000_000, dtype={'Lat': str, 'Long': str}):
        c = ch[ch['Lat'].notna() & (ch['Lat'] != 'NULL')]
        for cl, code, nm, la, lo in c.drop_duplicates(['ClusterName', 'StopCode'])[['ClusterName', 'StopCode', 'StopName', 'Lat', 'Long']].itertuples(index=False):
            loc.setdefault((cl, code), (float(la), float(lo), nm))
        for (cl, has), n in ch.groupby(['ClusterName', ch['Lat'].notna() & (ch['Lat'] != 'NULL')]).size().items():
            v = cl_rows.setdefault(cl, [0, 0]); v[0] += int(n); v[1] += int(n) if has else 0
located = pd.DataFrame([(k[0], k[1], v[0], v[1], v[2]) for k, v in loc.items()], columns=['ClusterName', 'StopCode', 'Lat', 'Long', 'StopName'])
located['TAZ'] = tag_points(located).values
north_share = located.groupby('ClusterName')['TAZ'].apply(lambda t: t.notna().mean())
northern_clusters = set(north_share[north_share >= 0.5].index)
print(f'(cluster, code) pairs with their own coordinates: {len(located):,}, inside the polygons {located["TAZ"].notna().sum():,}; clusters judged northern (≥ 50 % of located stops inside): {len(northern_clusters)} of {len(north_share)}  [{time.time() - t0:.0f} s]')
print('northern clusters:', sorted(northern_clusters))
KEY_TAZ = {(r.ClusterName, r.StopCode): int(r.TAZ) for r in located.itertuples() if r.TAZ == r.TAZ}       # located inside
KEY_OUT = {(r.ClusterName, r.StopCode) for r in located.itertuples() if r.TAZ != r.TAZ}                 # located outside
def key_to_taz(cluster, code):
    """TAZ of a (cluster, code): own coordinates first; else the north file for northern clusters; else None."""
    k = (cluster, code)
    if k in KEY_TAZ: return KEY_TAZ[k]
    if k in KEY_OUT: return None
    if cluster in northern_clusters and code in north_file.index and north_file.at[code, 'TAZ'] == north_file.at[code, 'TAZ']: return int(north_file.at[code, 'TAZ'])
    return None
located.to_csv(f'{OUT}/stops_located_by_cluster_2025.csv', index=False, float_format='%.6f')
north_file.reset_index().to_csv(f'{OUT}/stops_north_file_taz.csv', index=False, float_format='%.6f')

north stops file: 12,153 unique codes, 12,149 inside a TAZ_North polygon  [12 s]


(cluster, code) pairs with their own coordinates: 15,262, inside the polygons 12,822; clusters judged northern (≥ 50 % of located stops inside): 21 of 68  [48 s]
northern clusters: ['הגליל', 'העמקים', 'חדרה פרברי', 'חדרה-נתניה', 'חיפה עירוני', 'חיפה פרברי', 'חיפה-שרון-ירושלים', 'ירושלים צפון-ציר מזרחי', 'כרמיאל עירוני כרמיאל-חיפה חיפה-טבריה', 'מטרונית חיפה', "מרחב נצרת ג'י.בי טורס", 'מתמ"ז-קריות', 'קווי נצרת - נסיעות ותיירות', 'קווי נצרת – שאמ', 'קריית שמונה עירוני', 'קריית שמונה-חיפה', 'רכבל חיפה', 'רמת הגולן', 'תל אביב-גליל עמקים', 'תל אביב-חדרה', 'תל אביב-שרון-חיפה']


## 2. Reading the extracts: what is in them

The national bus file carries **every operator cluster in the country**, the rail cluster and
the Metronit cluster included (the same taps as the two dedicated files). The bus layer is
therefore the bus file **restricted to (cluster, code) stops located inside the polygons and to
clusters other than rail, Metronit and the two light-rail systems**; the Metronit and rail
layers come from their own files, the rail one restricted to the stations located in the north
by their own coordinates. Taps that cannot be placed are counted and dropped. Passenger counts are summed as signed
values (−1 rows are refunds); on the bus and Metronit rows a count of 0 is a tap that carried no
passenger and adds nothing; on the rail rows a count of 0 is an **exit-gate tap** (§5).

In [4]:
def read_agg(path, keep_cluster=None, drop_cluster=None, north_only=True, rail=False):
    """One pass over an extract. Returns per (date, stop, cluster, transfer[, hour]) net passengers and taps; for rail also entries / exits and the card-matched pairs."""
    rows, hours, pairs, ex = [], [], [], []
    t0 = time.time(); n_all = n_kept = 0
    for ch in pd.read_csv(path, usecols=USECOLS, chunksize=2_000_000, dtype={'TransactionTime': str, 'Lat': str, 'Long': str}):
        n_all += len(ch)
        assert (ch['Weekday'] == 3).all()
        ch = ch[ch['TransactionTime'].str.slice(0, 2).isin(HOURS)]
        if keep_cluster is not None: ch = ch[ch['ClusterName'].isin(keep_cluster)]
        if drop_cluster is not None: ch = ch[~ch['ClusterName'].isin(drop_cluster)]
        ch = ch.copy(); ch['TAZ'] = [key_to_taz(c_, s_) for c_, s_ in zip(ch['ClusterName'], ch['StopCode'])]
        if north_only: ch = ch[ch['TAZ'].notna()]
        n_kept += len(ch)
        if rail:
            ch = ch.copy(); ch['kind'] = np.where(ch['PassengersNumber'] > 0, 'entry', np.where(ch['PassengersNumber'] == 0, 'exit', 'refund'))
            ex.append(ch.groupby(['TransactionDate', 'StopCode', 'kind']).agg(pax=('PassengersNumber', 'sum'), taps=('StopCode', 'size')).reset_index())
            pairs.append(ch[['CardIDbi', 'StopCode', 'TransactionDate', 'TransactionTime', 'PassengersNumber']])
        else:
            ch = ch.copy(); ch['hour'] = ch['TransactionTime'].str.slice(0, 2); ch['q'] = ch['TransactionTime'].str.slice(3, 5).astype(int) // 15
            rows.append(ch.groupby(['TransactionDate', 'StopCode', 'ClusterName', 'TAZ', 'JourneyTransfer']).agg(pax=('PassengersNumber', 'sum'), taps=('StopCode', 'size')).reset_index())
            hours.append(ch.groupby(['TransactionDate', 'TAZ', 'JourneyTransfer', 'hour', 'q'])['PassengersNumber'].sum().reset_index())
    print(f'{os.path.basename(path)}: {n_all:,} rows read, {n_kept:,} kept  [{time.time() - t0:.0f} s]')
    if rail:
        return pd.concat(ex).groupby(['TransactionDate', 'StopCode', 'kind']).sum().reset_index(), pd.concat(pairs)
    return (pd.concat(rows).groupby(['TransactionDate', 'StopCode', 'ClusterName', 'TAZ', 'JourneyTransfer']).sum().reset_index(),
            pd.concat(hours).groupby(['TransactionDate', 'TAZ', 'JourneyTransfer', 'hour', 'q']).sum().reset_index())

# the national file: which clusters, and how much of it is in the north
clusters = {}
for ch in pd.read_csv(f'{RK}/Buses_RavKav.csv', usecols=['ClusterName', 'StopCode', 'PassengersNumber'], chunksize=2_000_000):
    ch['north'] = [key_to_taz(c_, s_) is not None for c_, s_ in zip(ch['ClusterName'], ch['StopCode'])]
    for (c, n), g in ch.groupby(['ClusterName', 'north']):
        k = (c, n); v = clusters.get(k, [0, 0.0]); v[0] += len(g); v[1] += float(g['PassengersNumber'].sum()); clusters[k] = v
cl = pd.DataFrame([(c, n, v[0], v[1]) for (c, n), v in clusters.items()], columns=['cluster', 'in north', 'rows', 'net passengers'])
cl_w = cl.pivot_table(index='cluster', columns='in north', values='net passengers', aggfunc='sum').fillna(0).rename(columns={True: 'in north', False: 'outside'})
cl_w['total'] = cl_w.sum(axis=1); cl_w = cl_w.sort_values('in north', ascending=False)
cl_w.to_csv(f'{OUT}/bus_file_clusters.csv', float_format='%.0f')
print('national bus file — operator clusters with taps at northern stops (net passengers over all dates, 06–09):')
print(cl_w[cl_w['in north'] > 0].round(0).to_string())

national bus file — operator clusters with taps at northern stops (net passengers over all dates, 06–09):
in north                                outside   in north      total
cluster                                                              
חיפה עירוני                               174.0  1351988.0  1352162.0
רכבת ישראל                            2242715.0  1120367.0  3363082.0
מטרונית חיפה                                0.0   656835.0   656835.0
העמקים                                    300.0   647288.0   647588.0
הגליל                                     552.0   467814.0   468366.0
מתמ"ז-קריות                                 0.0   419199.0   419199.0
חדרה-נתניה                              51129.0   318921.0   370050.0
קווי נצרת – שאמ                             0.0   239976.0   239976.0
חדרה פרברי                                 55.0   215156.0   215211.0
כרמיאל עירוני כרמיאל-חיפה חיפה-טבריה       23.0   138507.0   138530.0
קווי נצרת - נסיעות ותיירות                316.0   1153

In [5]:
bus_rows, bus_hours = read_agg(f'{RK}/Buses_RavKav.csv', drop_cluster={RAIL_CLUSTER, METRONIT_CLUSTER} | LRT_CLUSTERS)
met_rows, met_hours = read_agg(f'{RK}/Metronit_RavKav_Data.csv', keep_cluster={METRONIT_CLUSTER})
rail_ex, rail_taps = read_agg(f'{RK}/Rail_RavKav_Data.csv', keep_cluster={RAIL_CLUSTER}, north_only=False, rail=True)
# the rail extract's own coordinates are unreliable (Holon, Kfar Chabad and Netanya stations carry Binyamina's location, Nahariya carries Lev HaMifratz's):
# station locations come from the national GTFS stops table (codes 17000–17999), a committed copy standing in when the archive is only an LFS pointer
import zipfile, io
GTFS = 'Input/GTFS/israel-public-transportation.zip'; RAIL_CSV = f'{OUT}/rail_stations_gtfs_north.csv'
if not is_pointer(GTFS):
    gs = pd.read_csv(io.BytesIO(zipfile.ZipFile(GTFS).read('stops.txt')), dtype=str)
    gs['StopCode'] = pd.to_numeric(gs['stop_code'], errors='coerce'); gs = gs[gs['StopCode'].between(17000, 17999)].drop_duplicates('StopCode')
    gs = gs.rename(columns={'stop_name': 'StopName'}).assign(Lat=gs['stop_lat'].astype(float), Long=gs['stop_lon'].astype(float))
    gs['TAZ'] = tag_points(gs[['StopCode', 'StopName', 'Lat', 'Long']].reset_index(drop=True)).values
    gs = gs[gs['TAZ'].notna()][['StopCode', 'StopName', 'Lat', 'Long', 'TAZ']].rename(columns={'Lat': 'stop_lat', 'Long': 'stop_lon'}); gs['StopCode'] = gs['StopCode'].astype(int)
    gs.to_csv(RAIL_CSV, index=False); src = 'GTFS archive'
else:
    gs = pd.read_csv(RAIL_CSV); src = 'committed GTFS extract'
station_taz = {int(r.StopCode): int(r.TAZ) for r in gs.itertuples() if r.StopCode in set(rail_ex['StopCode'])}
rail_stations_north = sorted(station_taz)
print(f'rail: {rail_ex["StopCode"].nunique()} stations nationally, {len(rail_stations_north)} inside the TAZ_North polygons by their GTFS location ({src}); the extract\'s own coordinates place {sum(1 for (cl, c) in KEY_TAZ if cl == RAIL_CLUSTER)} inside, several of them wrongly')

Buses_RavKav.csv: 14,673,438 rows read, 4,157,214 kept  [57 s]


Metronit_RavKav_Data.csv: 655,470 rows read, 655,467 kept  [3 s]


Rail_RavKav_Data.csv: 6,016,201 rows read, 6,016,201 kept  [24 s]


rail: 70 stations nationally, 20 inside the TAZ_North polygons by their GTFS location (GTFS archive); the extract's own coordinates place 22 inside, several of them wrongly


## 3. One representative Tuesday

Each dataset's net boardings per date are compared with the median date; a date below 70 % of
the median is dropped (Passover week, the June 2025 war days, the autumn holidays, a few dates
with a handful of rows). The average over the retained dates is the **representative Tuesday**:
each stop's boardings summed over the retained dates and divided by their number, so a stop
with no tap on some dates is averaged as zero there.

In [6]:
def rep_dates(daily, label):
    med = daily.median(); keep = daily[daily >= REP_DAY_FLOOR * med].index
    print(f'{label}: {len(daily)} dates, median {med:,.0f} net boardings; kept {len(keep)}, dropped {sorted(set(daily.index) - set(keep))}')
    return sorted(keep), med
daily = pd.DataFrame({'bus (north, non-rail, non-Metronit)': bus_rows.groupby('TransactionDate')['pax'].sum(),
                      'Metronit': met_rows.groupby('TransactionDate')['pax'].sum(),
                      'rail entries (north stations)': rail_ex[(rail_ex['kind'] == 'entry') & rail_ex['StopCode'].isin(rail_stations_north)].groupby('TransactionDate')['pax'].sum()}).fillna(0)
KEEP = {}; MED = {}
for c in daily.columns: KEEP[c], MED[c] = rep_dates(daily[c], c)
daily['kept (all three)'] = daily.index.isin(sorted(set(KEEP[daily.columns[0]]) & set(KEEP[daily.columns[1]]) & set(KEEP[daily.columns[2]])))
daily.to_csv(f'{OUT}/daily_totals_by_date.csv', float_format='%.0f')
DATES = sorted(daily.index[daily['kept (all three)']]); print(f'representative Tuesdays common to the three datasets: {len(DATES)} ({DATES[0]} … {DATES[-1]})')

fig, ax = plt.subplots(figsize=(12, 4.2))
x = pd.to_datetime(daily.index)
for c, col in zip(daily.columns[:3], [BLUE, AQUA, PURPLE]):
    ax.plot(x, daily[c] / MED[c], 'o-', ms=3.5, lw=1, color=col, label=c)
ax.axhline(REP_DAY_FLOOR, color=ORANGE, lw=1, ls='--', label=f'floor: {REP_DAY_FLOOR:.0%} of the median date')
ax.set_ylabel('net boardings ÷ median date'); ax.set_ylim(0, 1.5); ax.grid(color=GRID, lw=0.6); ax.legend(frameon=False, fontsize=8, ncol=2)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.set_title('RavKav 2025, Tuesdays 06:00–09:00: daily net boardings relative to the median date, and the representative-day floor', loc='left', fontsize=10)
fig.savefig('Output/figures/ravkav_2025_daily_totals.png', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()

bus (north, non-rail, non-Metronit): 52 dates, median 92,690 net boardings; kept 42, dropped ['2025-04-15', '2025-06-17', '2025-06-24', '2025-08-05', '2025-08-12', '2025-08-19', '2025-08-26', '2025-09-23', '2025-10-07', '2025-10-14']
Metronit: 52 dates, median 14,122 net boardings; kept 46, dropped ['2025-04-15', '2025-06-17', '2025-06-24', '2025-09-23', '2025-10-07', '2025-10-14']
rail entries (north stations): 52 dates, median 22,872 net boardings; kept 46, dropped ['2025-04-15', '2025-06-17', '2025-06-24', '2025-09-23', '2025-10-07', '2025-10-14']
representative Tuesdays common to the three datasets: 42 (2025-01-07 … 2025-12-30)


In [7]:
def per_day(rows, dates, keys):
    r = rows[rows['TransactionDate'].isin(dates)]
    return (r.groupby(keys)[['pax', 'taps']].sum() / len(dates)).reset_index()
bus_d = per_day(bus_rows, DATES, ['StopCode', 'ClusterName', 'TAZ', 'JourneyTransfer']); bus_d['mode'] = 'bus'
met_d = per_day(met_rows, DATES, ['StopCode', 'ClusterName', 'TAZ', 'JourneyTransfer']); met_d['mode'] = 'metronit'
bm = pd.concat([bus_d, met_d]); bm['TAZ'] = bm['TAZ'].astype(int)
bm['flag'] = np.where(bm['JourneyTransfer'] == TRANSFER, 'transfer', 'nontransfer')
bm_stop = bm.pivot_table(index=['mode', 'ClusterName', 'StopCode', 'TAZ'], columns='flag', values='pax', aggfunc='sum').fillna(0).reset_index()
bm_stop['all'] = bm_stop['nontransfer'] + bm_stop['transfer']
bm_stop.to_csv(f'{OUT}/boardings_by_stop_2025.csv', index=False, float_format='%.3f')
by_taz = bm.pivot_table(index='TAZ', columns=['mode', 'flag'], values='pax', aggfunc='sum').fillna(0)
by_taz.columns = [f'{m}_{f}' for m, f in by_taz.columns]
for m in ['bus', 'metronit']: by_taz[f'{m}_all'] = by_taz[f'{m}_nontransfer'] + by_taz[f'{m}_transfer']
by_taz['transit_nontransfer'] = by_taz['bus_nontransfer'] + by_taz['metronit_nontransfer']; by_taz['transit_all'] = by_taz['bus_all'] + by_taz['metronit_all']
tot = by_taz.sum()
print('representative Tuesday, 06:00–09:00, boardings at stops inside TAZ_North (net passengers):')
print(pd.DataFrame({'non-transfer (journey origins)': [tot['bus_nontransfer'], tot['metronit_nontransfer'], tot['transit_nontransfer']], 'transfer boardings': [tot['bus_transfer'], tot['metronit_transfer'], tot['bus_transfer'] + tot['metronit_transfer']], 'all boardings': [tot['bus_all'], tot['metronit_all'], tot['transit_all']]}, index=['bus', 'Metronit', 'bus + Metronit']).round(0).to_string())
print(f"transfer share of boardings: bus {tot['bus_transfer'] / tot['bus_all']:.1%}, Metronit {tot['metronit_transfer'] / tot['metronit_all']:.1%}; TAZs with boardings: {len(by_taz)}")
print('operators in the bus layer (representative Tuesday, net passengers):'); print(bus_d.groupby('ClusterName')['pax'].sum().sort_values(ascending=False).round(0).head(15).to_string())

representative Tuesday, 06:00–09:00, boardings at stops inside TAZ_North (net passengers):
                non-transfer (journey origins)  transfer boardings  all boardings
bus                                    88728.0              2911.0        91639.0
Metronit                               13078.0              1029.0        14107.0
bus + Metronit                        101806.0              3940.0       105746.0
transfer share of boardings: bus 3.2%, Metronit 7.3%; TAZs with boardings: 733
operators in the bus layer (representative Tuesday, net passengers):
ClusterName
חיפה עירוני                             29646.0
העמקים                                  14256.0
הגליל                                   10222.0
מתמ"ז-קריות                              9168.0
חדרה-נתניה                               6980.0
קווי נצרת – שאמ                          5086.0
חדרה פרברי                               4792.0
כרמיאל עירוני כרמיאל-חיפה חיפה-טבריה     3031.0
קווי נצרת - נסיעות ותיירות           

## 4. Bus + Metronit OD from the journey origins and the OnBoard pattern

As in step 9: each TAZ's **journey origins** (its non-transfer boardings, bus + Metronit) are
distributed over the destinations by the OnBoard survey's P(alight | board at this TAZ),
renormalised over the known destinations. Origins the OnBoard survey does not cover take the
destination pattern of the May 2022 RavKav journeys from the same TAZ (`bus_od_taz_avg.csv`,
the step-8 product), and where neither exists the volume is reported as unallocated. A second
matrix uses **all boardings** (transfer legs included): it is the leg-level product, comparable
with the 2022 leg counts and with link loads, and it double-counts journeys with transfers by
construction. The OnBoard pattern's own unit (leg or journey, §8 caveat 7) is unchanged by this
step.

In [8]:
ob = pd.read_excel('Input/6_9_BusProbability_ByTAZ.xlsx')
known = ob.dropna(subset=['toTAZ']).copy(); known['toTAZ'] = known['toTAZ'].astype(int)
P = known.pivot_table(index='fromTAZ', columns='toTAZ', values='Probability', aggfunc='sum').fillna(0)
rs = P.sum(axis=1); P = P[rs > 0].div(rs[rs > 0], axis=0); P.index = P.index.astype(int)
rk22 = pd.read_csv('Output/bus/bus_od_taz_avg.csv', index_col=0); rk22.index = rk22.index.astype(int); rk22.columns = rk22.columns.astype(int)
P22 = rk22.div(rk22.sum(axis=1).replace(0, np.nan), axis=0).dropna(how='all')
DEST = sorted(set(P.columns) | set(P22.columns))

def build_od(vol, label):
    od = pd.DataFrame(0.0, index=sorted(vol.index), columns=DEST); src = {'OnBoard': 0.0, 'RavKav 2022 pattern': 0.0, 'unallocated': 0.0}; unalloc = {}
    for o, v in vol.items():
        if v <= 0: continue
        if o in P.index: od.loc[o, P.columns] += v * P.loc[o].values; src['OnBoard'] += v
        elif o in P22.index: od.loc[o, P22.columns] += v * P22.loc[o].values; src['RavKav 2022 pattern'] += v
        else: src['unallocated'] += v; unalloc[o] = v
    od.index.name = 'orig_taz'
    print(f'{label}: {vol.sum():,.0f} boardings → {od.values.sum():,.0f} allocated: OnBoard pattern {src["OnBoard"]:,.0f} ({src["OnBoard"] / vol.sum():.1%}), 2022 RavKav pattern {src["RavKav 2022 pattern"]:,.0f} ({src["RavKav 2022 pattern"] / vol.sum():.1%}), unallocated {src["unallocated"]:,.0f} in {len(unalloc)} TAZs')
    return od, pd.Series(unalloc)
od_j, un_j = build_od(by_taz['transit_nontransfer'], 'journeys (non-transfer boardings)')
od_l, un_l = build_od(by_taz['transit_all'], 'legs (all boardings)')
od_j.to_csv(f'{OUT}/bus_od_taz_2025.csv', float_format='%.6g'); od_l.to_csv(f'{OUT}/bus_od_taz_2025_legs.csv', float_format='%.6g')
by_taz.to_csv(f'{OUT}/boardings_by_taz_2025.csv', float_format='%.3f')
if len(un_j): print('largest unallocated origins (journeys):', un_j.sort_values(ascending=False).head(8).round(0).to_dict())

journeys (non-transfer boardings): 101,806 boardings → 101,680 allocated: OnBoard pattern 94,620 (92.9%), 2022 RavKav pattern 7,060 (6.9%), unallocated 127 in 14 TAZs


legs (all boardings): 105,746 boardings → 105,618 allocated: OnBoard pattern 98,321 (93.0%), 2022 RavKav pattern 7,297 (6.9%), unallocated 128 in 14 TAZs


largest unallocated origins (journeys): {333: 52.0, 3318: 39.0, 3808: 13.0, 302: 8.0, 1721: 8.0, 3829: 3.0, 226: 2.0, 3430: 1.0}


## 5. Rail: entries, exits and the station-to-station matrix

On the rail rows `PassengersNumber` is 1 on 59 % of the taps and 0 on 41 %. The 0-taps are the
**exit gates**: on a test date two thirds of the cards with an entry tap have a later 0-tap the
same morning, 98 % of them at a different station, a median 45 minutes later — a train ride.
The remaining third exit after 09:00, outside the extract. The rail OD is therefore built from
the **matched entry → next-exit pairs per card and date**, averaged over the representative
Tuesdays, and **expanded per origin station** by entries ÷ matched entries (the assumption: the
exits that fall after 09:00 spread like the matched ones — longer trips are more likely to be
among them, so the expansion slightly understates the long-distance destinations). Both ends
are kept nationally; the products give the northern stations' rows, the within-north block,
and the TAZ-level matrix on the station TAZs.

In [9]:
rt = rail_taps[rail_taps['TransactionDate'].isin(DATES)].copy()
rt['t'] = pd.to_timedelta(rt['TransactionTime'] + ':00'); rt = rt.sort_values(['TransactionDate', 'CardIDbi', 't'])
ent = rt[rt['PassengersNumber'] > 0]; exi = rt[rt['PassengersNumber'] == 0]
# first entry per card-date, then the first exit after it
e1 = ent.groupby(['TransactionDate', 'CardIDbi']).first().reset_index()[['TransactionDate', 'CardIDbi', 'StopCode', 't', 'PassengersNumber']].rename(columns={'StopCode': 'o', 't': 't_o'})
m = e1.merge(exi[['TransactionDate', 'CardIDbi', 'StopCode', 't']].rename(columns={'StopCode': 'd', 't': 't_d'}), on=['TransactionDate', 'CardIDbi'], how='left')
m = m[(m['t_d'] > m['t_o'])].sort_values(['TransactionDate', 'CardIDbi', 't_d']).groupby(['TransactionDate', 'CardIDbi']).first().reset_index()
m['min'] = (m['t_d'] - m['t_o']).dt.total_seconds() / 60
n_e = len(e1); n_m = len(m)
print(f'rail, {len(DATES)} representative Tuesdays: {n_e:,} card-date entries (first entry per card), {n_m:,} matched with a later exit ({n_m / n_e:.1%}); same-station pairs {(m["o"] == m["d"]).mean():.1%} (dropped); median ride {m["min"].median():.0f} min')
m = m[m['o'] != m['d']]
pairs = m.groupby(['o', 'd'])['PassengersNumber'].sum() / len(DATES)
entries = e1.groupby('o')['PassengersNumber'].sum() / len(DATES)          # first entries per card-date, per station, per day
entries_all = rail_ex[(rail_ex['kind'] == 'entry') & rail_ex['TransactionDate'].isin(DATES)].groupby('StopCode')['pax'].sum() / len(DATES)
matched_o = pairs.groupby(level=0).sum()
expand = (entries_all / matched_o).reindex(pairs.index.get_level_values(0)).values
rail_od = (pairs * expand).unstack().fillna(0)
rail_od.index.name = 'orig_station'
st_names = pd.read_csv(f'{RK}/Rail_RavKav_Data.csv', usecols=['StopCode', 'StopName'], nrows=3_000_000).drop_duplicates('StopCode').set_index('StopCode')['StopName']
north = [s for s in rail_od.index if s in rail_stations_north]
rail_od.to_csv(f'{OUT}/rail_od_station_2025_national.csv', float_format='%.4g')
rn = rail_od.loc[north]; rn.to_csv(f'{OUT}/rail_od_station_2025_north_origins.csv', float_format='%.4g')
rnn = rail_od.reindex(index=north, columns=north).fillna(0); rnn.to_csv(f'{OUT}/rail_od_station_2025_north.csv', float_format='%.4g')
exp_tab = pd.DataFrame({'station': [st_names.get(s, s) for s in north], 'entries per day (all)': entries_all.reindex(north).values, 'matched exits per day': matched_o.reindex(north).fillna(0).values}).set_index(pd.Index(north, name='StopCode'))
exp_tab['expansion'] = exp_tab['entries per day (all)'] / exp_tab['matched exits per day']; exp_tab['exits per day (as destination)'] = rail_ex[(rail_ex['kind'] == 'exit') & rail_ex['TransactionDate'].isin(DATES)].groupby('StopCode')['taps'].sum().reindex(north).fillna(0).values / len(DATES)
exp_tab['TAZ'] = [station_taz.get(s, np.nan) for s in north]
exp_tab.to_csv(f'{OUT}/rail_stations_north_2025.csv', float_format='%.2f')
print(f'northern stations: {len(north)}; entries per representative day {exp_tab["entries per day (all)"].sum():,.0f}; trips from northern stations in the expanded OD {rn.values.sum():,.0f}, of which to northern stations {rnn.values.sum():,.0f} ({rnn.values.sum() / rn.values.sum():.1%})')
print(exp_tab.sort_values('entries per day (all)', ascending=False).round(1).to_string())
# TAZ-level rail matrix on the station TAZs (both ends north)
rt_taz = rnn.copy(); rt_taz.index = [station_taz[s] for s in rt_taz.index]; rt_taz.columns = [station_taz[s] for s in rt_taz.columns]
rt_taz = rt_taz.groupby(level=0).sum().T.groupby(level=0).sum().T; rt_taz.index.name = 'orig_taz'
rt_taz.to_csv(f'{OUT}/rail_od_taz_2025.csv', float_format='%.4g')
tr19 = pd.read_csv('Output/train/train_od_taz_6_9.csv', index_col=0); tr19.index = tr19.index.astype(int); tr19.columns = tr19.columns.astype(int)
common = sorted(set(tr19.index) & set(rt_taz.index))
print(f'against the 2019 smartcard station matrix (step 10): 2019 {tr19.values.sum():,.0f} trips between {len(tr19)} station TAZs (× 0.793 = {tr19.values.sum() * 0.793:,.0f} at the 2022 level); 2025 {rt_taz.values.sum():,.0f} between {len(rt_taz)} (both ends north); on the {len(common)} common station TAZs 2019 {tr19.loc[common, common].values.sum():,.0f} vs 2025 {rt_taz.loc[common, common].values.sum():,.0f}')
a, b = tr19.loc[common, common].values.ravel(), rt_taz.loc[common, common].values.ravel()
print(f'cell correlation 2019 vs 2025 on the common block: r = {np.corrcoef(a, b)[0, 1]:.3f}')

rail, 42 representative Tuesdays: 3,010,021 card-date entries (first entry per card), 1,941,519 matched with a later exit (64.5%); same-station pairs 2.1% (dropped); median ride 46 min


northern stations: 20; entries per representative day 23,408; trips from northern stations in the expanded OD 23,408, of which to northern stations 9,437 (40.3%)
                  station  entries per day (all)  matched exits per day  expansion  exits per day (as destination)   TAZ
StopCode                                                                                                                
17024             בנימינה                 3371.7                 2133.6        1.6                           419.8  3907
17028           חדרה מערב                 2884.7                 2054.5        1.4                           240.4  4010
17020           חוף הכרמל                 2743.7                 1429.2        1.9                          1285.8  1517
17014               נהריה                 2260.3                 1424.1        1.6                           329.1   106
17117         קרית מוצקין                 1823.1                 1071.2        1.7                           363

## 6. Against 2022: the RavKav products of steps 8–9 and the survey-based 2022 layers

Comparisons by TAZ, by the 28 sub-areas of `Input/Submatrix_tazs.xlsx` and by the 25 V2 areas.
The 2022 RavKav boardings of step 8 are leg counts (every boarding of a journey), so they are
compared with the 2025 *all-boardings* column; the 2022 journey OD (94,203 journeys with both
ends in the study area) with the 2025 journey OD. The survey-based bus layer of step 16
(`bus_2022_area.csv`, calibrated, 2022 vintage) is the third reference.

In [10]:
ba22 = pd.read_csv('Output/bus/bus_boardings_alightings_taz.csv', index_col=0); ba22.index = ba22.index.astype(int)
od22 = pd.read_csv('Output/bus/bus_od_taz_new.csv', index_col=0); od22.index = od22.index.astype(int); od22.columns = od22.columns.astype(int)
cmp = pd.DataFrame({'boardings 2022 (legs, May 2022)': ba22['avg_boardings'], 'boardings 2025 (all)': by_taz['transit_all'], 'journeys 2025 (non-transfer)': by_taz['transit_nontransfer'], 'Metronit 2025 (all)': by_taz['metronit_all']}).fillna(0)
cmp['ratio 2025 / 2022 (legs)'] = cmp['boardings 2025 (all)'] / cmp['boardings 2022 (legs, May 2022)'].replace(0, np.nan)
cmp.to_csv(f'{OUT}/boardings_taz_2022_vs_2025.csv', float_format='%.2f')
print(f"study-area boardings, representative day: 2022 legs {cmp['boardings 2022 (legs, May 2022)'].sum():,.0f} → 2025 all boardings {cmp['boardings 2025 (all)'].sum():,.0f} (× {cmp['boardings 2025 (all)'].sum() / cmp['boardings 2022 (legs, May 2022)'].sum():.2f}); 2025 journey origins {cmp['journeys 2025 (non-transfer)'].sum():,.0f}")
print(f"journey OD, both ends in the study area: 2022 (RavKav journeys × OnBoard) {od22.values.sum():,.0f} → 2025 {od_j.values.sum():,.0f} (× {od_j.values.sum() / od22.values.sum():.2f}); legs 2025 {od_l.values.sum():,.0f}")
print('largest TAZs by 2025 boardings:'); print(cmp.sort_values('boardings 2025 (all)', ascending=False).head(10).round(0).to_string())

sub = pd.read_excel('Input/Submatrix_tazs.xlsx'); t2a = sub.set_index('TAZ')['AggAreaCode']; AREAS = sorted(sub['AggAreaCode'].unique()); legend = sub.drop_duplicates('AggAreaCode').set_index('AggAreaCode')['AggAreaName']
def to_area(mx, key, areas):
    long = mx.stack().reset_index(); long.columns = ['o', 'd', 'v']; long['O'] = long['o'].map(key); long['D'] = long['d'].map(key)
    return long.dropna(subset=['O', 'D']).groupby(['O', 'D'])['v'].sum().unstack().reindex(index=areas, columns=areas, fill_value=0).fillna(0)
a25, a25l, a22 = to_area(od_j, t2a, AREAS), to_area(od_l, t2a, AREAS), to_area(od22, t2a, AREAS)
bus22 = pd.read_csv('Output/ths2017/three_mode_2022/bus_2022_area.csv', index_col=0); bus22.index = bus22.index.astype(int); bus22.columns = bus22.columns.astype(int); bus22 = bus22.reindex(index=AREAS, columns=AREAS, fill_value=0)
a25.index.name = a25l.index.name = 'AggAreaCode'
a25.to_csv(f'{OUT}/bus_od_area_2025.csv', float_format='%.6g'); a25l.to_csv(f'{OUT}/bus_od_area_2025_legs.csv', float_format='%.6g')
area_cmp = pd.DataFrame({'area': legend.reindex(AREAS).values, 'origins RavKav 2022': a22.sum(axis=1).values, 'origins RavKav 2025 journeys': a25.sum(axis=1).values, 'origins RavKav 2025 legs': a25l.sum(axis=1).values, 'origins survey bus 2022 (step 16)': bus22.sum(axis=1).values}, index=AREAS)
area_cmp['2025 / 2022 RavKav'] = area_cmp['origins RavKav 2025 journeys'] / area_cmp['origins RavKav 2022'].replace(0, np.nan)
area_cmp['2025 RavKav / survey 2022'] = area_cmp['origins RavKav 2025 journeys'] / area_cmp['origins survey bus 2022 (step 16)'].replace(0, np.nan)
area_cmp.to_csv(f'{OUT}/bus_origins_area_2022_vs_2025.csv', float_format='%.2f')
print(f'28 sub-areas, both ends inside: RavKav 2022 {a22.values.sum():,.0f}, RavKav 2025 journeys {a25.values.sum():,.0f} (legs {a25l.values.sum():,.0f}), survey bus 2022 {bus22.values.sum():,.0f}')
print(area_cmp.sort_values('origins RavKav 2025 journeys', ascending=False).round(2).to_string())
r_cells = np.corrcoef(a22.values.ravel(), a25.values.ravel())[0, 1]; print(f'cell correlation of the 28-area journey matrices, 2022 vs 2025: r = {r_cells:.3f}')

xl = pd.ExcelFile('Input/Corridor_TAZ_Agg_V2.xlsx'); v2 = xl.parse('TazAgg').set_index('TAZ')['AggCode']; V2 = list(xl.parse('AreaCodes')['AggCode'])
v25, v25l, v22 = to_area(od_j, v2, V2), to_area(od_l, v2, V2), to_area(od22, v2, V2)
v25.index.name = v25l.index.name = 'AggCode'; v25.to_csv(f'{OUT}/bus_od_area_v2_2025.csv', float_format='%.6g'); v25l.to_csv(f'{OUT}/bus_od_area_v2_2025_legs.csv', float_format='%.6g')
print(f'25 V2 areas, both ends inside: RavKav 2022 {v22.values.sum():,.0f} → 2025 journeys {v25.values.sum():,.0f} (× {v25.values.sum() / v22.values.sum():.2f}), legs {v25l.values.sum():,.0f}')

fig, ax = plt.subplots(figsize=(7.5, 7))
lim = max(area_cmp['origins RavKav 2022'].max(), area_cmp['origins RavKav 2025 journeys'].max()) * 1.05
ax.scatter(area_cmp['origins RavKav 2022'], area_cmp['origins RavKav 2025 journeys'], s=40, color=BLUE, alpha=0.7)
for a_, r_ in area_cmp.iterrows(): ax.annotate(r_['area'], (r_['origins RavKav 2022'], r_['origins RavKav 2025 journeys']), fontsize=7, color=INK2, xytext=(3, 3), textcoords='offset points')
ax.plot([0, lim], [0, lim], '--', color=INK2, lw=1); ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel('journey origins, RavKav May 2022 × OnBoard (28 sub-areas)'); ax.set_ylabel('journey origins, RavKav 2025 (non-transfer boardings) × OnBoard'); ax.grid(color=GRID, lw=0.6)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.set_title('Bus + Metronit journey origins by sub-area: 2022 against 2025', loc='left', fontsize=10)
fig.savefig('Output/figures/ravkav_2025_vs_2022_area_origins.png', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()

study-area boardings, representative day: 2022 legs 157,264 → 2025 all boardings 105,746 (× 0.67); 2025 journey origins 101,806
journey OD, both ends in the study area: 2022 (RavKav journeys × OnBoard) 94,203 → 2025 101,680 (× 1.08); legs 2025 105,618
largest TAZs by 2025 boardings:
      boardings 2022 (legs, May 2022)  boardings 2025 (all)  journeys 2025 (non-transfer)  Metronit 2025 (all)  ratio 2025 / 2022 (legs)
1219                           6478.0                3381.0                        2994.0               1511.0                       1.0
1517                           3703.0                1665.0                        1483.0                418.0                       0.0
4015                           1813.0                1518.0                        1420.0                  0.0                       1.0
1311                           1790.0                1286.0                        1140.0                695.0                       1.0
1104                           

28 sub-areas, both ends inside: RavKav 2022 23,995, RavKav 2025 journeys 25,677 (legs 27,082), survey bus 2022 25,858
                     area  origins RavKav 2022  origins RavKav 2025 journeys  origins RavKav 2025 legs  origins survey bus 2022 (step 16)  2025 / 2022 RavKav  2025 RavKav / survey 2022
25  Kiryat Motzkin-Bialik              5354.63                       6115.10                   6393.18                            6666.94                1.14                       0.92
24             Kiryat Yam              4117.03                       3306.24                   3397.97                            4217.14                0.80                       0.78
13              Hamifrats              1079.11                       2266.23                   2559.02                             243.12                2.10                       9.32
28       Kiryat Ata South              2080.37                       1935.09                   1992.41                            2328.95     

## 6b. Boarding-hour peak factors (task B1e)

The taps carry the minute, so the boarding-time profile within 06:00–09:00 is binned in 15-minute
intervals and the peak-hour factors of step 20 (PHF₃ₕ = busiest 60 minutes ÷ three hours;
PHF₆₀ = busiest 60 minutes ÷ 4 × busiest 15 minutes) are computed on **boardings** — the
independent check on the survey departure-time factors of §6r / §6y (bus 0.589 study-area-wide,
network 0.549 up / 0.457 down). Three scopes: the whole study area, the corridor TAZs of the V2
aggregation, and the ten trunk station areas.

In [11]:
def phf(prof):
    s = np.asarray(prof, float); tot = s.sum()
    if tot <= 0: return dict(PHF3h=np.nan, start=np.nan, PHF60=np.nan)
    p = s / tot; win = np.array([p[i:i + 4].sum() for i in range(len(p) - 3)]); k = int(np.argmax(win))
    return dict(PHF3h=win[k], start=6 + k / 4, PHF60=win[k] / (4 * p[k:k + 4].max()))
hh = pd.concat([bus_hours.assign(mode='bus'), met_hours.assign(mode='metronit')]); hh = hh[hh['TransactionDate'].isin(DATES)]
hh['bin'] = (hh['hour'].astype(int) - 6) * 4 + hh['q']
trunk_taz = set(v2[v2.isin(range(201, 211))].index); corridor_taz = set(v2.index)
rows = []
for scope, mask in [('study area', hh['TAZ'].notna()), ('V2 corridor TAZs', hh['TAZ'].isin(corridor_taz)), ('trunk station areas 201–210', hh['TAZ'].isin(trunk_taz))]:
    for mode in ['bus', 'metronit', 'bus + metronit']:
        for flag, fm in [('all boardings', slice(None)), ('non-transfer', NONTRANSFER)]:
            g = hh[mask & (hh['mode'].isin(['bus', 'metronit']) if mode == 'bus + metronit' else hh['mode'] == mode)]
            if flag == 'non-transfer': g = g[g['JourneyTransfer'] == NONTRANSFER]
            prof = g.groupby('bin')['PassengersNumber'].sum().reindex(range(12)).fillna(0).values
            st = phf(prof); rows.append({'scope': scope, 'mode': mode, 'boardings': flag, 'per day': prof.sum(), **st, 'peak hour': f"{int(st['start']):02d}:{int((st['start'] % 1) * 60):02d}–{int(st['start'] + 1):02d}:{int((st['start'] % 1) * 60):02d}" if st['start'] == st['start'] else ''})
phf_tab = pd.DataFrame(rows); phf_tab.to_csv(f'{OUT}/boarding_hour_peak_factors_2025.csv', index=False, float_format='%.4f')
print(phf_tab.round(3).to_string(index=False))
prof_all = hh.groupby(['mode', 'bin'])['PassengersNumber'].sum().unstack(0).reindex(range(12)).fillna(0) / len(DATES)
prof_all.index = [f'{6 + b // 4:02d}:{(b % 4) * 15:02d}' for b in prof_all.index]; prof_all.to_csv(f'{OUT}/boarding_profile_15min_2025.csv', float_format='%.1f')

                      scope           mode     boardings  per day  PHF3h  start  PHF60   peak hour
                 study area            bus all boardings  3848846  0.475   7.25  0.912 07:15–08:15
                 study area            bus  non-transfer  3726581  0.477   7.25  0.910 07:15–08:15
                 study area       metronit all boardings   592491  0.433   7.25  0.938 07:15–08:15
                 study area       metronit  non-transfer   549289  0.433   7.25  0.936 07:15–08:15
                 study area bus + metronit all boardings  4441337  0.470   7.25  0.915 07:15–08:15
                 study area bus + metronit  non-transfer  4275870  0.471   7.25  0.913 07:15–08:15
           V2 corridor TAZs            bus all boardings   974446  0.459   7.25  0.864 07:15–08:15
           V2 corridor TAZs            bus  non-transfer   931063  0.461   7.25  0.860 07:15–08:15
           V2 corridor TAZs       metronit all boardings   508399  0.433   7.25  0.941 07:15–08:15
          

## 7. What this layer is, and what re-anchoring the chain on it would take

**What it is.** A 2025 boarding layer for the study area on a representative Tuesday,
06:00–09:00, by stop and TAZ, with journey origins and transfer legs told apart; a bus +
Metronit journey OD on the OnBoard pattern (the step-9 construction on 2025 volumes) and a leg
OD beside it; a rail station-to-station OD measured from the entry and exit taps; the operator
clusters actually present in the extract (the question task B1c asked the provider); and
boarding-hour peak factors (task B1e).

**What it is not.** Alightings for bus and Metronit are still the OnBoard survey's pattern, whose
unit is unconfirmed (§8 caveat 7); the rail expansion assumes the after-09:00 exits spread like
the matched ones; the representative day is an average over the retained Tuesdays, not a single
observed day, and the July–August dates (school holidays, 15–25 % below the spring level) are
inside it.

**Re-anchoring.** Steps 15–16 anchor the bus layer's volumes on the May 2022 journeys
(`Output/bus/bus_od_taz_new.csv`) and read the 2022 leg boardings for the within-superzone
origin split. Pointing them at `bus_od_taz_2025.csv` and `boardings_by_taz_2025.csv` moves the
bus anchor to 2025 while the car, taxi-type and rail layers stay grown to 2022 — a vintage
mismatch unless the whole base moves to 2025 (the car layer grown 2018 → 2025 on the BU-2025
zonal file that exists, rail from `rail_od_taz_2025.csv` instead of the national ratio). That is a
base-year decision, recorded in METHODOLOGY §6af; the products here are ready for it.